<a href="https://colab.research.google.com/github/Saadhvi-29/GEN_AI_LAB/blob/main/GenAI_Linkedin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import datetime

# ===============================
# STEP 1: SYSTEM PROMPT
# ===============================

SYSTEM_PROMPT = """
You are a simple AI agent.
Your goal is to answer user questions using stored memory.
You can retrieve relevant past information to respond.
"""

# ===============================
# STEP 2: EMBEDDING MODEL (LOCAL)
# ===============================

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ===============================
# STEP 3: MEMORY (FAISS VECTOR STORE)
# ===============================

memory_texts = []
faiss_index = None

def add_to_memory(text):
    """Store text into agent memory"""
    memory_texts.append(text)

def build_faiss_index():
    """Build FAISS index from memory"""
    global faiss_index
    embeddings = embedding_model.encode(memory_texts)
    dimension = embeddings.shape[1]
    faiss_index = faiss.IndexFlatL2(dimension)
    faiss_index.add(np.array(embeddings))

def retrieve_memory(query, k=1):
    """Retrieve relevant memory"""
    if faiss_index is None:
        return None
    query_embedding = embedding_model.encode([query])
    distances, indices = faiss_index.search(np.array(query_embedding), k)
    return memory_texts[indices[0][0]]

# ===============================
# STEP 4: TOOLS
# ===============================

def get_current_time():
    return str(datetime.datetime.now())

TOOLS = {
    "time": get_current_time
}

# ===============================
# STEP 5: ORCHESTRATION LOGIC
# ===============================

def agent_think(user_input):
    """Agent reasoning logic"""

    # Tool usage
    if "time" in user_input.lower():
        return TOOLS["time"]()

    # Memory retrieval
    retrieved = retrieve_memory(user_input)
    if retrieved:
        return f"I remember this: {retrieved}"

    return "I do not have enough information yet."

# ===============================
# STEP 6: USER INTERFACE (CLI)
# ===============================

def run_agent():
    print("AI Agent started. Type 'exit' to stop.\n")

    # Initial memory
    add_to_memory("AI agents use memory to improve responses")
    add_to_memory("FAISS enables fast similarity search")
    add_to_memory("Tools help agents perform actions")

    build_faiss_index()

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Agent stopped.")
            break

        response = agent_think(user_input)
        print("Agent:", response)

# ===============================
# STEP 7: OBSERVABILITY
# ===============================

if __name__ == "__main__":
    run_agent()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AI Agent started. Type 'exit' to stop.

You: What is the time right now?
Agent: 2026-02-09 15:46:51.474011
You: How do u remember it all?
Agent: I remember this: AI agents use memory to improve responses
You: EXIT
Agent stopped.
